In [ ]:
# KV-Cache Reuse Exploration
#
# **Setup:** Qwen3-0.6B and Qwen3-1.7B both have 28 layers, same KV shape [8, seq, 128].
# CKA shows high similarity for first 6 and last 10 layers.
#
# **Finding:** Keys have high cross-model cosine similarity (~0.95) but values have
# near-zero similarity (~0.06-0.10). Freezing both K+V produces garbage; freezing
# only keys works because attention = softmax(Q * K^T) * V, and V stays in the
# small model's representational space.
#
# **Approach:** Run the small model's prefill, but at frozen layers replace the
# computed keys with the large model's keys. Values are always computed by the
# small model. Non-frozen layers see the frozen keys through attention.
#
# **Goal:** Measure whether injecting large-model keys at selected layers improves
# the small model's HumanEval accuracy.

In [10]:
import re
import signal
import json
from pathlib import Path
from contextlib import contextmanager

import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache

## Configuration

In [26]:
MODEL_SMALL = "Qwen/Qwen3-0.6B"  # target model (runs generation)
MODEL_LARGE = "Qwen/Qwen3-1.7B"  # donor model (provides KV cache for selected layers)

# Layers to freeze from the large model during small model's prefill
# Both models have 28 layers (0-27), same KV shape [8, seq, 128]
REPLACE_LAYERS = list(range(0, 2))  # first 2 layers

MAX_NEW_TOKENS = 1024
NUM_SAMPLES = None  # None = all 164
TIMEOUT = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Small model: {MODEL_SMALL}")
print(f"Large model: {MODEL_LARGE}")
print(f"Frozen layers: {REPLACE_LAYERS}")
print(f"Device: {DEVICE}")

Small model: Qwen/Qwen3-0.6B
Large model: Qwen/Qwen3-1.7B
Frozen layers: [0, 1]
Device: cuda


## Utilities

In [27]:
class TimeoutError(Exception):
    pass


@contextmanager
def time_limit(seconds: int):
    def handler(signum, frame):
        raise TimeoutError("Timed out!")
    signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)


def extract_code(text: str) -> str:
    """Extract python code from model output (```python blocks or raw text)."""
    match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def check_correctness(problem: dict, code: str, timeout: int = 5) -> dict:
    """Run extracted code against HumanEval tests."""
    task_id = problem["task_id"]
    full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    try:
        with time_limit(timeout):
            exec(full_code, {})
        return {"task_id": task_id, "passed": True, "error": None}
    except TimeoutError:
        return {"task_id": task_id, "passed": False, "error": "Timed out"}
    except Exception as e:
        return {"task_id": task_id, "passed": False, "error": str(e)}


def format_prompt(problem_prompt: str) -> str:
    return (
        "Complete the following Python function. "
        "Return ONLY the complete function implementation in a python code block.\n\n"
        + problem_prompt
    )


def generate_from_kv(model, tokenizer, input_ids, past_key_values,
                     max_new_tokens=1024):
    """
    Generate using an externally-provided KV cache.

    Crop the cache to N-1 tokens, feed the Nth token as input_ids with
    explicit cache_position and an attention_mask covering all N positions
    (cached + current). This forces generate() to run a single-token
    "prefill" that fills the Nth KV slot, then the normal decode loop runs.
    """
    seq_len = past_key_values.get_seq_length()
    past_key_values.crop(seq_len - 1)
    last_token = input_ids[:, -1:]
    cache_position = torch.tensor([seq_len - 1], device=input_ids.device)
    # Attention mask must cover all positions: N-1 cached + 1 current
    attention_mask = torch.ones(1, seq_len, device=input_ids.device, dtype=torch.long)

    with torch.no_grad():
        out = model.generate(
            input_ids=last_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            cache_position=cache_position,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

## Load Models & Dataset

In [4]:
print("Loading tokenizers and models...")
tok_small = AutoTokenizer.from_pretrained(MODEL_SMALL, trust_remote_code=True)
model_small = AutoModelForCausalLM.from_pretrained(
    MODEL_SMALL, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True
).eval()

tok_large = AutoTokenizer.from_pretrained(MODEL_LARGE, trust_remote_code=True)
model_large = AutoModelForCausalLM.from_pretrained(
    MODEL_LARGE, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True
).eval()

print(f"Small model layers: {model_small.config.num_hidden_layers}")
print(f"Large model layers: {model_large.config.num_hidden_layers}")
print("Models loaded.")

Loading tokenizers and models...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████████████████████████| 2/2 [00:09<00:00,  4.61s/it]

Small model layers: 28
Large model layers: 28
Models loaded.


In [5]:
dataset = load_dataset("openai_humaneval", split="test")
if NUM_SAMPLES is not None:
    dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))
print(f"{len(dataset)} problems loaded")

164 problems loaded


## Sanity Check: Run One Problem

In [28]:
problem = dataset[0]
prompt = format_prompt(problem["prompt"])

messages = [{"role": "user", "content": prompt}]
input_text = tok_small.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs_small = tok_small(input_text, return_tensors="pt").to(DEVICE)
inputs_large = tok_large(input_text, return_tensors="pt").to(DEVICE)

# Prefill both models
with torch.no_grad():
    out_small = model_small(**inputs_small, use_cache=True)
    out_large = model_large(**inputs_large, use_cache=True)

kv_small = out_small.past_key_values
kv_large = out_large.past_key_values

print(f"Small model KV: {len(kv_small)} layers, shape: {kv_small[0][0].shape}")
print(f"Large model KV: {len(kv_large)} layers, shape: {kv_large[0][0].shape}")

# Cross-model KV compatibility check
print(f"\nCross-model KV compatibility (cosine similarity):")
print(f"{'Layer':>6} {'Keys cos':>10} {'Vals cos':>10} {'Keys range_s':>14} {'Keys range_l':>14}")
for li in range(0, 28, 3):
    ks, kl = kv_small[li][0].flatten().float(), kv_large[li][0].flatten().float()
    vs, vl = kv_small[li][1].flatten().float(), kv_large[li][1].flatten().float()
    cos_k = torch.nn.functional.cosine_similarity(ks.unsqueeze(0), kl.unsqueeze(0)).item()
    cos_v = torch.nn.functional.cosine_similarity(vs.unsqueeze(0), vl.unsqueeze(0)).item()
    print(f"{li:>6} {cos_k:>10.4f} {cos_v:>10.4f} [{ks.min():.1f},{ks.max():.1f}] [{kl.min():.1f},{kl.max():.1f}]")

Small model KV: 28 layers, shape: torch.Size([1, 8, 148, 128])
Large model KV: 28 layers, shape: torch.Size([1, 8, 148, 128])

Cross-model KV compatibility (cosine similarity):
 Layer   Keys cos   Vals cos   Keys range_s   Keys range_l
     0     0.9623     0.0815 [-493.5,517.5] [-337.0,394.0]
     3     0.9416     0.0688 [-97.4,94.8] [-105.9,92.2]
     6     0.8771     0.1174 [-80.8,91.1] [-79.9,91.0]
     9     0.8495     0.0766 [-38.7,41.9] [-42.9,41.7]
    12     0.8594     0.0549 [-55.0,58.2] [-50.1,58.8]
    15     0.9025     0.0846 [-61.6,70.5] [-55.2,65.9]
    18     0.6818     0.0985 [-28.4,30.7] [-26.6,29.0]
    21     0.6961     0.1143 [-34.5,28.0] [-33.2,27.2]
    24     0.6592     0.0819 [-44.9,39.9] [-52.2,36.2]
    27     0.7955     0.0981 [-39.6,42.4] [-42.9,44.6]


## Keys-Only Frozen Cache

At frozen layers, replace the small model's computed keys with the large model's
keys. Values are always computed by the small model so they stay in its
representational space. `FrozenKeysCache` intercepts the cache update to swap keys.

In [31]:
class FrozenKeysCache(DynamicCache):
    """
    Cache that freezes only keys at selected layers from a donor model.
    Values are always computed by the running model.
    """

    def __init__(self, frozen_layers: set[int]):
        super().__init__()
        self.frozen_layers = frozen_layers
        self.frozen_keys = {}  # layer_idx -> key tensor [1, nh, seq, hd]

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        if layer_idx in self.frozen_layers and layer_idx in self.frozen_keys:
            # Replace computed keys with donor keys; values pass through normally
            return super().update(self.frozen_keys[layer_idx], value_states, layer_idx, cache_kwargs)
        return super().update(key_states, value_states, layer_idx, cache_kwargs)


def build_frozen_keys_cache(kv_large, frozen_layers: list[int]) -> FrozenKeysCache:
    """Build a FrozenKeysCache with donor keys from the large model."""
    cache = FrozenKeysCache(set(frozen_layers))
    for li in frozen_layers:
        cache.frozen_keys[li] = kv_large[li][0]  # keys only
    return cache


# Test
frozen_cache = build_frozen_keys_cache(kv_large, REPLACE_LAYERS)
print(f"FrozenKeysCache: frozen layers {REPLACE_LAYERS}")
print(f"Donor key shape: {frozen_cache.frozen_keys[0].shape}")

FrozenKeysCache: frozen layers [0, 1]
Donor key shape: torch.Size([1, 8, 148, 128])


In [ ]:
# Small model prefill with frozen keys from large model
frozen_cache = build_frozen_keys_cache(kv_large, REPLACE_LAYERS)

with torch.no_grad():
    out_hybrid = model_small(**inputs_small, use_cache=True, past_key_values=frozen_cache)

hybrid_kv = out_hybrid.past_key_values
print(f"After prefill: cache has {hybrid_kv.get_seq_length()} tokens, {len(hybrid_kv)} layers")

# Generate
hybrid_text = generate_from_kv(
    model_small, tok_small, inputs_small["input_ids"],
    DynamicCache.from_legacy_cache(hybrid_kv.to_legacy_cache()),
    max_new_tokens=MAX_NEW_TOKENS,
)
hybrid_code = extract_code(hybrid_text)

# Baseline
with torch.no_grad():
    out_baseline = model_small.generate(
        **inputs_small, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tok_small.eos_token_id,
    )
baseline_text = tok_small.decode(out_baseline[0][inputs_small["input_ids"].shape[1]:], skip_special_tokens=True)
baseline_code = extract_code(baseline_text)

print(f"\nTask: {problem['task_id']}")
print(f"\n--- Baseline (small only) ---")
print(baseline_code[:500])
print(f"\n--- Hybrid (frozen keys at layers {REPLACE_LAYERS} from large) ---")
print(hybrid_code[:500])
print(f"\n--- Baseline test: {check_correctness(problem, baseline_code, TIMEOUT)}")
print(f"--- Hybrid test:   {check_correctness(problem, hybrid_code, TIMEOUT)}")

After prefill: cache has 148 tokens, 28 layers

Task: HumanEval/0

--- Baseline (small only) ---
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    
    Args:
        numbers: List of floats
        threshold: Float value to check against
    
    Returns:
        True if any two numbers in the list are closer to each other than the threshold, False otherwise.
    """
    numbers.sort()
    for i in range(1, len(numbers)):
        if

--- Hybrid (frozen keys at layers [0, 1] from large) ---
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """Check if in given list of numbers, are any two numbers closer to each other than the given threshold."""
    numbers.sort()
    for i in range(len(numbers) - 1):
        if abs(numbers[i] - numbers[i + 1]) < threshold:
            return Tr

## Full Evaluation

Run three configurations on all HumanEval problems:
1. **Small only** (0.6B baseline)
2. **Hybrid** (0.6B + 1.7B KV at selected layers)
3. **Large only** (1.7B baseline)

In [ ]:
def evaluate_model(model, tokenizer, dataset, label, max_new_tokens=1024,
                   timeout=5, device="cuda", kv_override_fn=None):
    """
    Evaluate a model on HumanEval.

    kv_override_fn: optional callable(inputs) -> DynamicCache
        If provided, uses generate_from_kv with the injected cache.
    """
    results = []
    passed = 0

    for i, problem in enumerate(dataset):
        prompt = format_prompt(problem["prompt"])
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            if kv_override_fn is not None:
                kv = kv_override_fn(inputs)
                generated_text = generate_from_kv(
                    model, tokenizer, inputs["input_ids"], kv,
                    max_new_tokens=max_new_tokens,
                )
            else:
                out = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
                generated_text = tokenizer.decode(
                    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
                )

        code = extract_code(generated_text)
        result = check_correctness(problem, code, timeout=timeout)
        results.append(result)
        if result["passed"]:
            passed += 1

        if (i + 1) % 20 == 0 or i == len(dataset) - 1:
            print(f"  [{label}] {i+1}/{len(dataset)} — running pass@1: {passed/(i+1)*100:.1f}%")

    accuracy = passed / len(dataset) * 100
    print(f"  [{label}] Final pass@1: {accuracy:.2f}% ({passed}/{len(dataset)})")
    return results, accuracy

In [19]:
# 1. Small model baseline
print("=== Evaluating small model (0.6B) ===")
results_small, acc_small = evaluate_model(
    model_small, tok_small, dataset, "0.6B",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
)

=== Evaluating small model (0.6B) ===
  [0.6B] Final pass@1: 0.61% (1/164)


In [ ]:
# 2. Hybrid: small model with frozen keys from large model
def hybrid_kv_fn(inputs_batch):
    """
    1. Prefill large model to get donor keys
    2. Build FrozenKeysCache with donor keys at frozen layers
    3. Run small model's forward — frozen layers use donor keys,
       values always computed by small model
    """
    with torch.no_grad():
        out_l = model_large(
            input_ids=inputs_batch["input_ids"],
            attention_mask=inputs_batch.get("attention_mask", None),
            use_cache=True,
        )
        frozen_cache = build_frozen_keys_cache(out_l.past_key_values, REPLACE_LAYERS)
        out_s = model_small(**inputs_batch, use_cache=True, past_key_values=frozen_cache)
    return DynamicCache.from_legacy_cache(out_s.past_key_values.to_legacy_cache())

print(f"=== Evaluating hybrid (0.6B + frozen keys at layers {REPLACE_LAYERS} from 1.7B) ===")
results_hybrid, acc_hybrid = evaluate_model(
    model_small, tok_small, dataset, "hybrid",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
    kv_override_fn=hybrid_kv_fn,
)

In [ ]:
# 3. Large model baseline
print("=== Evaluating large model (1.7B) ===")
results_large, acc_large = evaluate_model(
    model_large, tok_large, dataset, "1.7B",
    max_new_tokens=MAX_NEW_TOKENS, timeout=TIMEOUT, device=DEVICE,
)